In [ ]:
#@title Copyright 2019 Google LLC. { display-mode: "form" }
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="ee-notebook-buttons" align="left"><td>
<a target="_blank"  href="http://colab.research.google.com/github/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>
</td><td>
<a target="_blank"  href="https://github.com/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb"><img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" /> View source on GitHub</a></td></table>

[Instalacja Pythona](https://developers.google.com/earth-engine/guides/python_install?hl=pl)

# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [ ]:
import ee

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='playgroundgee')

## Test the API

Test the API by printing the elevation of Mount Everest.

In [ ]:
print(ee.String('Hello from the Earth Engine servers!').getInfo())

In [ ]:
# Print the elevation of Mount Everest.
dem = ee.Image('USGS/SRTMGL1_003')
xy = ee.Geometry.Point([86.9250, 27.9881])
elev = dem.sample(xy, 30).first().get('elevation').getInfo()
print('Mount Everest elevation (m):', elev)

## Map visualization

`ee.Image` objects can be displayed to notebook output cells. The following two
examples demonstrate displaying a static image and an interactive map.


### Static image

The `IPython.display` module contains the `Image` function, which can display
the results of a URL representing an image generated from a call to the Earth
Engine `getThumbUrl` function. The following cell will display a thumbnail
of the global elevation model.

In [ ]:
# Import the Image function from the IPython.display module.
from IPython.display import Image

# Display a thumbnail of global elevation.
Image(url = dem.updateMask(dem.gt(0))
  .getThumbURL({'min': 0, 'max': 4000, 'dimensions': 512,
                'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']}))

### Interactive map

The [geemap](https://github.com/gee-community/geemap)
library can be used to display `ee.Image` objects on an interactive
[ipyleaflet](https://github.com/jupyter-widgets/ipyleaflet) map.

The following cell provides an example of using the `geemap.Map` object to
display an elevation model.

In [ ]:
# Import the geemap library.
from ipyleaflet import Map, basemaps, basemap_to_tiles
import geemap

# Set visualization parameters.
vis_params = {
  'min': 0,
  'max': 4000,
  'palette': ['006633', 'E5FFCC', '662A00', 'D8CDBE', 'F5F5F5']}

# Create a map object.
m = geemap.Map(center=[20, 0], zoom=3)

# Add the elevation model to the map object.
m.add_ee_layer(dem.updateMask(dem.gt(0)), vis_params, 'DEM')

# Display the map.
display(m)

## Chart visualization

Some Earth Engine functions produce tabular data that can be plotted by
data visualization packages such as `matplotlib`. The following example
demonstrates the display of tabular data from Earth Engine as a scatter
plot. See [Charting in Colaboratory](https://colab.sandbox.google.com/notebooks/charts.ipynb)
for more information.

In [ ]:
# Import the matplotlib.pyplot module.
import matplotlib.pyplot as plt

# Fetch a Landsat TOA image.
img = ee.Image('LANDSAT/LT05/C02/T1_TOA/LT05_034033_20000913')

# Select Red and NIR bands and sample 500 points.
samp_fc = img.select(['B3','B4']).sample(scale=30, numPixels=500)

# Arrange the sample as a list of lists.
samp_dict = samp_fc.reduceColumns(ee.Reducer.toList().repeat(2), ['B3', 'B4'])
samp_list = ee.List(samp_dict.get('list'))

# Save server-side ee.List as a client-side Python list.
samp_data = samp_list.getInfo()

# Display a scatter plot of Red-NIR sample pairs using matplotlib.
plt.scatter(samp_data[0], samp_data[1], alpha=0.2)
plt.xlabel('Red', fontsize=12)
plt.ylabel('NIR', fontsize=12)
plt.show()

In [ ]:
import datetime
import math

Umieszczenie tekstu w kontenerze i wysłanie na serwer Earth Engine

In [ ]:
greetString = ee.String('Ahoy there!')
print(greetString)
print(greetString.getInfo())

Tworzenie obiektów liczbowych na serwerze

In [ ]:
serverNumber = ee.Number(math.e)
print(f'e = {serverNumber.getInfo():.2f}')


Tworzenie list

In [ ]:
list_numbers = [x for x in range(15) if x % 2 == 0]
list_words = ['sei','sette','otto','nove','dieci']
eeNumbers = ee.List(list_numbers)
eeWords = ee.List(list_words)

print(eeNumbers.getInfo())
#do liczenia
print(f'Value at index 2: {eeNumbers.getNumber(2).getInfo()}')

print(eeWords.getInfo())
#do działania na tekście
print(f'Value at index 1: {eeWords.getString(1).getInfo()}')

#jeszcze jest getArray(index) to jak są tablice zagnieżdżone, struktury wielowymiarowe

Tworzenie słowników

In [ ]:
dictionary = {'e': math.e, 'pi':math.pi, 'phi': (1 + math.sqrt(5)/2)}
eeDict = ee.Dictionary(dictionary)

print(f"Euler: {eeDict.getNumber('e').getInfo()}")
print(f"Pi: {eeDict.getNumber('pi').getInfo()}")
print(f"Golden Ratio: {eeDict.getNumber('phi').getInfo()}")

print(f"Dictionary keys: {eeDict.keys().getInfo()} <- zwraca wartość jako listę\nDIctionary values: {eeDict.values().getInfo()} <- tu też")

Reprezentacja czasu

In [ ]:
#najpierw imporcik datetime

date = datetime.date(2026, 3, 17)
date_long = datetime.datetime(2026,4,7,13,32,14)
date_today = datetime.date.today()
date_now = datetime.datetime.utcnow() #+ datetime.timedelta(hours=2)

print(date)
print(date_long)
print(date_today)
print(date_now)

eeDate = ee.Date(date_now)
print(eeDate.format('dd-MMM-YYYY HH:mm:ss').getInfo())

#daty przydaja się do filtrowania kolekcji jako argument filterDate()

In [ ]:
#print(ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318'))

image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318')
print(image.getInfo())
#możemy wybrać konkretne bandy które będa wyświetlane bo to jest raster aktualnie
#bandy są w słowniku
params = {
    'bands': ['B4','B3','B2'],
    'min': 0.02, #zakres wartości pixeli które mapujemy na skalę szarości/kolorów
    'max': 0.3, #to jest po prostu bezpieczny zakres dla landsata 0.02–0.4
    'gamma': 1.4 #u dwornika było bebecita <1 ciemno >1 jasno
    }

m = geemap.Map(basemap = basemaps.CartoDB.Voyager)
m.centerObject(image, 12) #poziom powiększenia
m.add_ee_layer(image, params)
display(m)

[Pasma Landsat](https://www.usgs.gov/faqs/what-are-band-designations-landsat-satellites)
[Wizualizacja](https://developers.google.com/earth-engine/guides/image_visualization?hl=pl)
[Kolekcje](https://developers.google.com/earth-engine/datasets?hl=pl)

In [ ]:
collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')

point = ee.Geometry.Point(-122.262, 37.8719)
start = ee.Date('2014-06-01')
finish = ee.Date('2014-10-01')

params = {
    'bands': ['B4','B3','B2'],
    'min': 0.02,
    'max': 0.4,
    'gamma': 1.4
}

filteredCollection = collection.filterBounds(point).filterDate(start,finish).sort('CLOUD_COVER',True)
first = filteredCollection.first()

m2 = geemap.Map(basemap = basemaps.CartoDB.Voyager)
m2.centerObject(point, 9)
m2.add_ee_layer(first, params)
display(m2)


In [ ]:
vectorCollection = ee.FeatureCollection('TIGER/2016/States')

names = vectorCollection.aggregate_array('NAME').sort()
print(names.getInfo())

filteredFC = vectorCollection.filter(ee.Filter.eq('NAME', 'California'))

m3 = geemap.Map(center=[37.798,-119.604], zoom = 6) #uwaga na odwrócone współrzędne, w gee/leaflet/geemap najpierw szerokosc potem dlugosc
m3.add_ee_layer(filteredFC,{'color': 'green', 'fillColor':'00000000', 'width': 2}, 'California') #defaultowe wartosci parametru
display(m3)



Obliczanie pasm

In [ ]:
def getNDVI(image):
  return image.normalizedDifference(['B4','B3'])

image1 = ee.Image('LANDSAT/LT05/C02/T1_TOA/LT05_044034_19900604')
image2 = ee.Image('LANDSAT/LT05/C02/T1_TOA/LT05_044034_20100611')

ndvi1 = getNDVI(image1)
ndvi2 = getNDVI(image2)

ndviDiff = ndvi2.subtract(ndvi1)
vis_params = {
    'min': -1.0,
    'max': 1.0,
    'palette':['#640000', '#ff0000', '#ffff00', '#00c800', '#006400']}

ndvi_diff_params = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['red', 'white', 'green']}

m4 = geemap.Map(center=[37.5,-122.5], zoom = 8)
m4.add_layer(ndvi1, vis_params, 'NDVI 1990')
m4.add_layer(ndvi2, vis_params, 'NDVI 2010')
m4.add_layer(ndviDiff, ndvi_diff_params, 'NDVI difference')
display(m4)

Mapowanie. NIe używać pętli for do przeglądania elementów w kolekcji -> nieefektywne. map() można zastosować do ImageCollection, FeatureCollection albo do List, argument funkcji to element kolekcji do któej jest mapowana

In [ ]:

#dodanie pasma NDVI do każdego obrazu w kolekcji
def addNDVI(image):
  return image.addBands(image.normalizedDifference(['B5','B4']))

collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA').filterBounds(ee.Geometry.Point(-122.262, 37.8719)).filterDate('2014-06-01', '2014-10-01')
ndviCollection = collection.map(addNDVI)
median_ndvi = ndviCollection.median()
best_cloud = ndviCollection.sort('CLOUD_COVER',True).first()

vis_params = {
    'bands': ['nd'],
    'min': -1.0,
    'max': 1.0,
    'palette':['#640000', '#ff0000', '#ffff00', '#00c800', '#006400']
}

print(collection.first().bandNames().getInfo())
#tu dodany kanał NDVI
print(ndviCollection.first().bandNames().getInfo())

print(ndviCollection.size().getInfo())

m5 = geemap.Map(basemap = basemaps.Esri.WorldImagery, center=[37.5,-122.5], zoom = 8)
m5.add_ee_layer(best_cloud, vis_params, 'NDVI')
display(m5)


Dodawanie nowych atrybutów

In [ ]:
def addField(feature):
  sum = ee.Number(feature.getNumber('property1')).add(feature.getNumber('property2'))
  return feature.set({'sum': sum})

features = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(-122.4536, 37.7403),
    {'property1': 100, 'property2': 100}),
    ee.Feature(ee.Geometry.Point(-118.2294, 34.039),
    {'property1': 200, 'property2': 300})
])

featureCollection = features.map(addField)

print(featureCollection.getInfo())

#wypisanie pól i ich wartośći
for f in featureCollection.getInfo()['features']:
  print(f['properties'])

#wypisanie współrzędnych tak o dla sportu
for f in featureCollection.getInfo()['features']:
  print(f['geometry']['coordinates'])

Zmiana typu, z kolekcji obrazów na kolekcję punktów na podstawie centroidów obrazów

Zawsze można sobie wyexportowaćdo qgisa i tam wystylizować kropeczki.

In [ ]:
def getGeom(image):
  return ee.Feature(image.geometry().centroid(), {
      'foo': 1,
      'date': image.date().format('YYYY-MM-dd')
      }) #foo to placeholder, pokazuje składnię to znaczy że można tam dodac cokolwiek, jakikolwiek argument np. 'date': image.date().format('YYYY-MM-dd')

collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA').filterBounds(ee.Geometry.Point(-122.262, 37.8719)).filterDate('2014-06-01', '2014-10-01')

featureCollection = ee.FeatureCollection(collection.map(getGeom))

print(featureCollection.getInfo())

m6 = geemap.Map(center=[37.47,-122.10], zoom = 15)
m6.add_ee_layer(featureCollection, {}, "Centroidy kolekcji zdjęć")
display(m6)

filterBounds() filtruje kolekcje do obrazów które przecinają lub zawierają podaną geometrię

In [ ]:
collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA').filterBounds(ee.Geometry.Point(-122.262, 37.8719)).filterDate('2014-01-01', '2014-12-31').sort('CLOUD_COVER')

#Compute the median of each pixel for each band of the 5 least cloudy scenes.
median = collection.limit(5).reduce(ee.Reducer.median())

# reduce() redukuje z kolekcji do jednego obrazu